In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve, validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings("ignore")

In [2]:
def load_and_preprocess_data(file_path):
    # Chargement des données
    dataset = pd.read_csv(file_path)
    
    # Supprimer la colonne ID si elle existe
    if 'ID' in dataset.columns:
        dataset = dataset.drop("ID", axis=1)
    
    # Traiter les valeurs manquantes
    for column in dataset.columns:
        if dataset[column].dtype != object:
            median_value = dataset[column].median()
            dataset[column] = dataset[column].fillna(median_value)
        else:
            most_frequent_value = dataset[column].value_counts().index[0]
            dataset[column] = dataset[column].fillna(most_frequent_value)
    
    return dataset


In [3]:
def encode_data(data):
    # Encodage des colonnes binaires
    binary_columns = [
        "H17A", "H17B", "H17C", "H17D", "H17E", "H17F", "H17G", "H17H", "H17I", "H17J",
        "H17Y", "H18A", "H18B", "H18C", "H18D", "H18E", "H18F", "H18G", "H18H", "H18I",
        "H18J", "H18Y", "H20A", "H20B", "H20C", "H20D", "H20E", "H20Y", "H21A", "H21B",
        "H21C", "H21D", "H21Y"
    ]
    for col in binary_columns:
        data[col] = data[col].map({"Oui": 1, "Non": 0})
    
    categorical_columns = ["Connexion", "TypeLogmt_1", "TypeLogmt_2", "TypeLogmt_3", "H08_Impute", "H09_Impute", "BoxLabel"]
    target = "Accès internet"  # Assure-toi que le nom de la colonne est correct

    # Encodage One-Hot
    one_hot_dataset = pd.get_dummies(data, columns=categorical_columns)

    # Frequency Encoding
    freq_encoded_dataset = data.copy()
    for col in categorical_columns:
        freq_encoded_dataset[col] = freq_encoded_dataset[col].map(data[col].value_counts(normalize=True).to_dict())

    # Target Encoding
    target_encoded_dataset = data.copy()
    for col in categorical_columns:
        if data[col].dtype == 'object':
            target_mean = data.groupby(col)[target].mean().to_dict()
            target_encoded_dataset[col] = target_encoded_dataset[col].map(target_mean)

    return one_hot_dataset, freq_encoded_dataset, target_encoded_dataset


In [4]:
def display_correlation(data, target_column, columns_to_exclude=[]):
    print(f"Calculating correlation with target column '{target_column}'...")
    data_corr = data.drop(columns=columns_to_exclude, errors='ignore')
    correlation = data_corr.corr()[target_column].abs().sort_values()
    print(correlation)
    
    plt.figure(figsize=(16, 8))
    sns.heatmap(data_corr.corr().abs(), cmap='coolwarm', annot=True)
    plt.show()
    print("Correlation heatmap displayed.")
    return correlation

def filter_columns_by_correlation(data, correlation, threshold_low=-0.2, threshold_high=0.2):
    cols_to_drop = correlation[(correlation >= threshold_low) & (correlation <= threshold_high)].index.tolist()
    print(f"Columns to drop due to low correlation: {cols_to_drop}")
    return data.drop(columns=cols_to_drop)

def apply_pca(data, n_components=10):
    scaler = StandardScaler()
    pca = PCA(n_components=n_components)
    pipeline = Pipeline([('scaler', scaler), ('pca', pca)])
    data_pca = pipeline.fit_transform(data)
    
    explained_variance = pca.explained_variance_ratio_
    print(f"Explained variance by each component: {explained_variance}")
    return data_pca, pca


In [5]:
def train_and_evaluate_models(X_train, X_test, y_train, y_test):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gradient Boosting": GradientBoostingClassifier(),
        "SVM": SVC(),
        "Naive Bayes": GaussianNB(),
        "Neural Network": MLPClassifier(max_iter=1000)
    }

    results = {}
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        results[model_name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1
        }
    
    return results

def print_results(results):
    for model_name, metrics in results.items():
        print(f"Results for {model_name}:")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")
        print("\n")


In [6]:
def perform_grid_search(X_train, y_train, model, param_grid):
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")
    return grid_search.best_estimator_


In [7]:
def save_model(model, filename):
    joblib.dump(model, filename)
    print(f"Model saved as {filename}")


In [ ]:
def main(file_path, batch_size=10000):
    try:
        dataset = load_and_preprocess_data(file_path)
        dataset = dataset.drop([col for col in dataset.columns if '.1' <= col <= '.3999'], axis=1)


        # Scinder le dataset 
        dataset = dataset.iloc[:len(dataset)//16]

        one_hot_dataset, freq_encoded_dataset, target_encoded_dataset = encode_data(dataset)









        print("Calculating initial correlation and filtering columns...")
        initial_correlation = display_correlation(dataset, "Accès internet", 
                                                  columns_to_exclude=["Connexion", "TypeLogmt_1", "TypeLogmt_2", "TypeLogmt_3", "H08_Impute", "H09_Impute", "BoxLabel"])
        dataset_filtered = filter_columns_by_correlation(dataset, initial_correlation)
        
        one_hot_dataset, freq_encoded_dataset, target_encoded_dataset = encode_data(dataset_filtered)
        
        print("Calculating and filtering correlations for One-Hot Encoded dataset...")
        one_hot_correlation = display_correlation(one_hot_dataset, "Accès internet")
        one_hot_dataset_filtered = filter_columns_by_correlation(one_hot_dataset, one_hot_correlation)

        print("Calculating and filtering correlations for Frequency Encoded dataset...")
        freq_encoded_correlation = display_correlation(freq_encoded_dataset, "Accès internet")
        freq_encoded_dataset_filtered = filter_columns_by_correlation(freq_encoded_dataset, freq_encoded_correlation)

        print("Calculating and filtering correlations for Target Encoded dataset...")
        target_encoded_correlation = display_correlation(target_encoded_dataset, "Accès internet")
        target_encoded_dataset_filtered = filter_columns_by_correlation(target_encoded_dataset, target_encoded_correlation)

        X_one_hot = one_hot_dataset_filtered.drop("Accès internet", axis=1)
        y_one_hot = one_hot_dataset_filtered["Accès internet"]

        print("Splitting data into training and testing sets...")
        X_train_one_hot, X_test_one_hot, y_train_one_hot, y_test_one_hot = train_test_split(X_one_hot, y_one_hot, test_size=0.2, random_state=42)
        print(f"Data split completed. Training set shape: {X_train_one_hot.shape}, Testing set shape: {X_test_one_hot.shape}")







        
        
#        display_correlation(one_hot_dataset, "Accès internet")
#        display_correlation(freq_encoded_dataset, "Accès internet")
#        display_correlation(target_encoded_dataset, "Accès internet")
        
#        X_one_hot = one_hot_dataset.drop("Accès internet", axis=1)
#        y_one_hot = one_hot_dataset["Accès internet"]

#        X_train_one_hot, X_test_one_hot, y_train_one_hot, y_test_one_hot = train_test_split(X_one_hot, y_one_hot, test_size=0.2, random_state=42)









        
        X_train_one_hot_pca, pca = apply_pca(X_train_one_hot)
        X_test_one_hot_pca = pca.transform(X_test_one_hot)

        results_one_hot = train_and_evaluate_models(X_train_one_hot_pca, X_test_one_hot_pca, y_train_one_hot, y_test_one_hot)
        print_results(results_one_hot)

        # Example of Grid Search for Random Forest
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
        best_rf_model = perform_grid_search(X_train_one_hot_pca, y_train_one_hot, RandomForestClassifier(), param_grid)
        save_model(best_rf_model, 'best_random_forest_model.pkl')

    except KeyError as e:
        print(f"Key error: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main("Train.csv")


Calculating initial correlation and filtering columns...
Calculating correlation with target column 'Accès internet'...
.2749    0.000399
.579     0.000522
.3473    0.000594
.1813    0.000652
.869     0.000753
           ...   
.3710         NaN
.3786         NaN
.3804         NaN
.3861         NaN
.3954         NaN
Name: Accès internet, Length: 4035, dtype: float64
